# 02 - Feature Engineering and Model Development

I prepare the binary target, preprocess numeric and categorical features according to the dataset description, compare Logistic Regression, Random Forest, and XGBoost with cross-validation, tune the best baseline, and evaluate it once on a held-out test set.

In [1]:
from pathlib import Path
import sys

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project root containing the src folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import download_dataset, load_dataset
from src.feature_engineering import build_preprocessor, prepare_features
from src.onnx_utils import convert_pipeline_to_onnx, save_onnx_model

DATASET_PATH = PROJECT_ROOT / "data" / "heart_disease.csv"

# Convert the original 0-4 target into binary target as given in the assignment instructions.
download_dataset(DATASET_PATH)
df = load_dataset(DATASET_PATH)
X, y = prepare_features(df)

print("Binary target counts:")
print(y.value_counts().sort_index())
print("\nBinary target proportions:")
print(y.value_counts(normalize=True).sort_index().round(3))

# Split before fitting preprocessing so test data does not influence imputation or scaling.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Binary target counts:
target
0    164
1    139
Name: count, dtype: int64

Binary target proportions:
target
0    0.541
1    0.459
Name: proportion, dtype: float64


The target is now binary, with original value 0 kept as negative and values 1-4 grouped as positive. Following the dataset description, age, trestbps, chol, thalach, oldpeak, and ca are numeric features. Sex, cp, fbs, restecg, exang, slope, and thal are categorical. Stratification preserves class proportions in the holdout split and CV folds. The preprocessor is fitted inside each training fold, so validation and test rows do not contribute to imputation or scaling values.

In [2]:
# Keep preprocessing inside each pipeline so every CV fold fits it only on that fold's training rows.
models = {
    "Logistic Regression": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", RandomForestClassifier(random_state=42)),
        ]
    ),
    "XGBoost": Pipeline(
        [
            ("preprocessor", build_preprocessor(X_train)),
            (
                "classifier",
                XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=1),
            ),
        ]
    ),
}

# Try a small grid for each classifier.
parameter_grids = {
    "Logistic Regression": {"classifier__C": [0.01, 0.1, 1, 10]},
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [None, 8],
        "classifier__min_samples_leaf": [1, 2],
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [2, 3],
        "classifier__learning_rate": [0.03, 0.1],
    },
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
}

# Tune every model using the same five stratified folds and compare each model's best combination.
searches = {}
comparison_rows = []
for name, model in models.items():
    search = GridSearchCV(
        model,
        parameter_grids[name],
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        n_jobs=1,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best_index = search.best_index_
    comparison_rows.append(
        {
            "model": name,
            "cv_accuracy": search.cv_results_["mean_test_accuracy"][best_index],
            "cv_precision": search.cv_results_["mean_test_precision"][best_index],
            "cv_recall": search.cv_results_["mean_test_recall"][best_index],
            "cv_roc_auc": search.best_score_,
            "best_parameters": search.best_params_,
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values("cv_roc_auc", ascending=False)
display(comparison.round(3))

# Select the best model and hyperparameter combination by mean CV ROC-AUC.
selected_name = comparison.iloc[0]["model"]
best_search = searches[selected_name]
best_model = best_search.best_estimator_
print(f"Selected model: {selected_name}")
print(f"Best parameters: {best_search.best_params_}")

,model,cv_accuracy,cv_precision,cv_recall,cv_roc_auc,best_parameters
0,Logistic Regression,0.851,0.878,0.792,0.911,{'classifier__C': 0.1}
1,Random Forest,0.802,0.800,0.765,0.897,"{'classifier__max_depth': 8, 'classifier__min_..."
2,XGBoost,0.822,0.828,0.783,0.885,"{'classifier__learning_rate': 0.03, 'classifie..."


Selected model: Logistic Regression
Best parameters: {'classifier__C': 0.1}


For each classifier, the grid search tests every listed hyperparameter combination using the same five stratified folds. The table reports CV metrics and best parameters for each family; the overall selection uses mean CV ROC-AUC. The held-out test set remains separate for final evaluation.

In [3]:
# The model and hyperparameter combination were selected by the CV searches above.
# Evaluate the selected, already-tuned pipeline once on the untouched test set.
onnx_model = convert_pipeline_to_onnx(best_model, list(X_train.columns))
onnx.checker.check_model(onnx_model)

test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)[:, 1]
onnx_session = ort.InferenceSession(
    onnx_model.SerializeToString(), providers=["CPUExecutionProvider"]
)
onnx_inputs = {
    feature: X_test[[feature]].to_numpy(dtype=np.float32)
    for feature in X_test.columns
}
onnx_predictions = onnx_session.run(None, onnx_inputs)[0]
if not np.array_equal(test_predictions, onnx_predictions):
    raise RuntimeError("ONNX predictions do not match the selected champion pipeline.")
print(f"Selected model: {selected_name}")
print("Best parameters:", best_search.best_params_)
print(f"Best cross-validation ROC-AUC: {best_search.best_score_:.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, test_predictions):.3f}")
print(f"Test precision: {precision_score(y_test, test_predictions, zero_division=0):.3f}")
print(f"Test recall:    {recall_score(y_test, test_predictions, zero_division=0):.3f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, test_probabilities):.3f}")
print("Test confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

# Save the complete pipeline so inference applies the same preprocessing as training.
MODEL_PATH = PROJECT_ROOT / "artifacts" / "heart_disease_model.onnx"
save_onnx_model(onnx_model, MODEL_PATH)
print(f"Saved ONNX model to: {MODEL_PATH}")

Selected model: Logistic Regression
Best parameters: {'classifier__C': 0.1}
Best cross-validation ROC-AUC: 0.911
Test accuracy:  0.885
Test precision: 0.839
Test recall:    0.929
Test ROC-AUC:   0.961
Test confusion matrix:
[[28  5]
 [ 2 26]]
Saved ONNX model to: d:\M.Tech BITS Pilani WILP\Sem 3\MLOps\MLOps Labs\MLOps-Assignment\artifacts\heart_disease_model.onnx


The grid search evaluates multiple hyperparameter combinations for all three classifier families using five-fold stratified cross-validation. In this run, Logistic Regression had the highest mean CV ROC-AUC (0.911), followed by Random Forest (0.897) and XGBoost (0.885); the selected Logistic Regression setting was C=0.1. On the held-out test set, the selected pipeline achieved accuracy 0.885, precision 0.839, recall 0.929, and ROC-AUC 0.961, with confusion matrix [[28, 5], [2, 26]]. The selected preprocessing and classifier are exported together as `artifacts/heart_disease_model.onnx` for portable deployment. Because this is a small dataset and one train/test split, these results are not evidence of clinical performance.